In [2]:
#imports
import os
import json
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import sqlite3

In [3]:
load_dotenv(override=True)
gemini_api_key = os.getenv("GOOGLE_API_KEY")
Model = "gemini-2.5-flash"
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",api_key=gemini_api_key)
ollama = OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")

In [4]:
DB = "carrental.db"
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS cars(
    car_type TEXT PRIMARY KEY,
    price_per_day REAL,
    total_units Integer
    )
    ''')
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS booking(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_name TEXT,
    car_type TEXT,
    date TEXT,
    days Integer
    )
    ''')
    conn.commit()

In [5]:
car_data = [('sedan',40,3),('suv',70,2),('truck',90,1),('convertible',120,2)]
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.executemany(
        "INSERT OR IGNORE INTO cars (car_type,price_per_day,total_units) VALUES (?,?,?)",car_data
    )
    conn.commit()

In [6]:
def get_car_price(car_type,days):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('Select price_per_day FROM cars where car_type = ?',(car_type.lower(),))
        result = cursor.fetchone()
    
    if result :
        total = result[0]*days
        return f"A {car_type} costs ${result[0]}/day. for {days} days that's {total} total."
    else:
        return f"Sorry we dont have the {car_type} available."

In [7]:
def check_availability(car_type, date):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('Select count(*) FROM booking WHERE car_type = ? AND date = ?',(car_type.lower(),date))
        booked = cursor.fetchone()[0]
        cursor.execute('Select total_units FROM cars WHERE car_type = ?',(car_type.lower(),))
        result = cursor.fetchone()
        if not result :
            return f"We don't have {car_type} in our fleet"
        total = result[0]
        if booked < total:
            return f"Yes, {car_type} is available on {date}"
        else:
            return f"No, {car_type} is not available on {date}"

In [12]:
def make_booking(customer_name,car_type,date,days):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        availability = check_availability(car_type,date)
        if "No" in availability or "don't" in availability:
            return availability
        with sqlite3.connect(DB) as conn:
            cursor = conn.cursor()
            cursor.execute(
                "INSERT INTO booking (customer_name,car_type,date,days) VALUES (?,?,?,?)",
                (customer_name,car_type.lower(),date,days)
            )
            conn.commit()
        return f"Booking confirmed for {customer_name}! {car_type} booked on {date} for {days} day"

In [ ]:
get_car_price_function = {
    "name": "get_car_price",
    "description":"Get the rental price for a specific car type for a given number",
    "parameters":{
        "type":"object",
        "properties":{
            "car_type":{
                "type":"string",
                "description":"The type of car e.g. sedan,suv,truck,convertible"
            },
            "days":{
                "type":"integer",
                "description":"The number of days customer wants to rent the car"
            }
        },
        "required":["car_type","days"],
        "additionalProperties":False
    }
}

check_availability_function = {
    "name":"check_availability",
    "description":"Checks if the prefered car is available or not",
    "parameters":{
        "type":"object",
        "properties":{
            "car_type":{
                "type":"string",
                "description":"The type of car e.g. sedan,suv,truck,convertible"
            },
            "date":{
                "type":"string",
                "description":"Date in YYYY-MM-DD format e.g. 2025-08-09"
            }
        },
        "required":["car_type","date"],
        "additionalProperties":False
    }
}

make_booking_function = {
    "name":"make_booking",
    "description":"Make booking for customers",
    "parameters":{
        "type":"object",
        "properties":{
            "customer_name":{
                "type":"string",
                "description":"Name of the customer e.g. Aarjav,Jhon"
            },
            "car_type":{
                "type":"string",
                "description":"The type of car e.g. sedan, suv, truck, convertible"
            },
            "date":{
                "type":"string",
                "description":"Date in YYYY-MM-DD format e.g. 2026-12-01"
            },
            "days":{
                "type":"integer",
                "description":"Number of days customer wants to rent the car"
            }
        },
        "required":["customer_name","car_type","date","days"],
        "additionalProperties":False
    }
}

tools = [{"type":"function","function":get_car_price_function},{"type":"function","function":check_availability_function},{"type":"function","function":make_booking_function}]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (6137290.py, line 64)

In [13]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_car_price":
            arguments = json.loads(tool_call.function.arguments)
            car_type = arguments.get("car_type")
            days = arguments.get("days")
            result = get_car_price(car_type,days)
            responses.append(
                {
                    "role":"tool",
                    "content":result,
                    "tool_call_id":tool_call.id
                }
            )
        elif tool_call.function.name == "check_availability":
            arguments = json.loads(tool_call.function.arguments)
            car_type = arguments.get("car_type")
            date = arguments.get("date")
            result = check_availability(car_type,date)
            responses.append(
                {
                    "role":"tool",
                    "content":result,
                    "tool_call_id":tool_call.id
                }
            )
        elif tool_call.function.name == "make_booking":
            arguments = json.loads(tool_call.function.arguments)
            customer_name = arguments.get("customer_name")
            car_type = arguments.get("car_type")
            date = arguments.get("date")
            days = arguments.get("days")
            result = make_booking(customer_name,car_type,date,days)
            responses.append(
                {
                    "role":"tool",
                    "content":result,
                    "tool_call_id":tool_call.id
                }
            )
    return responses

In [11]:
system_prompt = """
You are a AI Assistant at a Car Rental called DriveEasy.
Give short, courteous answers, no more than 1 sentance.
Always be Accuraate. If you dont know the answer, say so.
"""

In [ ]:
def chat(message,history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    response = gemini.chat.completions.create(
        model=Model,
        messages=messages,
        tools = tools
    )
    while response.choices[0].finish_reason == "tool_calls" :
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=Model,messages=messages,tools=tools)

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat).launch()